# ARI5121 – Applied NLP | Speech Analysis and Evaluation Project

## 0. Setup

In [72]:
import time

# File handling
import os
import shutil
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd

# Audio loading
import librosa

# Deep learning / Hugging Face
import torch
import torch.nn.functional as F
from transformers import AutoFeatureExtractor, AutoModelForAudioXVector

# Evaluation and visualisation
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

### 0.1. Config Parameters

In [73]:
MODEL_NAME = "microsoft/wavlm-base-plus-sv"
TARGET_SAMPLE_RATE = 16000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f">> Device Selected: {DEVICE}")

CORPUS_DIR = Path("../ABI-1 Corpus")
OUTPUT_DIR = Path("../extracted_wavs")
RESULTS_DIR = Path("../results")

>> Device Selected: cuda


### 0.2. Execution Timer

In [74]:
start_time = time.time()

## 1. Validate Corpus Existance

In [75]:
# validate corpus
def validate_corpus_folder(corpus_dir):
    """
    Checks that the ABI-1 corpus folder exists inside the project directory.
    """

    if not corpus_dir.exists():
        raise FileNotFoundError(
            f">> [ERROR] Corpus folder not found.\n"
            f"Expected location:\n{corpus_dir.resolve()}\n\n"
            f"Make sure the ABI-1 corpus folder is inside your project directory."
        )

    if not corpus_dir.is_dir():
        raise NotADirectoryError(
            f">> [ERROR] Corpus path exists, but it is not a folder:\n{corpus_dir.resolve()}"
        )

    print(f">> [OK] Corpus folder found: {corpus_dir.resolve()}")

In [76]:
validate_corpus_folder(CORPUS_DIR)

>> [OK] Corpus folder found: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/SpeechProcessing/ABI-1 Corpus


## 2. Extract shortpassage WAV files

In [77]:
def extract_shortpassage_waves(corpus_dir, output_dir):
    """
    Copies only the required shortpassage .wav files into a cleaned folder.
    """
    
    # create output dir if it doesn't exits
    output_dir.mkdir(parents=True, exist_ok=True)
    
    copied_count = 0
    skipped_count = 0
    
    for root, _, files in os.walk(corpus_dir):
        root_path = Path(root)
        
        for file_name in files:
            file_path = root_path / file_name
            
            # Check is a file is a .wav file and has shortpassage in the name
            is_wav = file_path.suffix.lower() == ".wav"
            is_shortpassage = "shortpassage" in file_name.lower()
            
            if not is_wav or not is_shortpassage:
                skipped_count += 1
                continue
            
            # Get path relative to the original corpus folder
            relative_path = file_path.relative_to(corpus_dir)

            # Copy into cleaned folder while preserving folder structure
            destination_path = output_dir / relative_path

            # Create destination parent folders
            destination_path.parent.mkdir(parents=True, exist_ok=True)

            # Copy file
            shutil.copy2(file_path, destination_path)

            copied_count += 1
            # print(f">> [COPIED] {relative_path}")

    print(">> --- [CLEANING COMPLETE] --- :")
    print(f">> WAV files copied: {copied_count}")
    print(f">> Files skipped: {skipped_count}")
    print(f">> Cleaned folder: {output_dir.resolve()}")

In [78]:
extract_shortpassage_waves(CORPUS_DIR, OUTPUT_DIR)

>> --- [CLEANING COMPLETE] --- :
>> WAV files copied: 855
>> Files skipped: 7702
>> Cleaned folder: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/SpeechProcessing/extracted_wavs


## 3. Create a metadata CSV

In [ ]:
def create_metadata(cleaned_dir):
    """
    Creates a metadata CSV from the cleaned WAV folder.

    Each row stores:
    - accent
    - gender
    - speaker_id
    - file_path
    """

    records = []

    for wav_path in cleaned_dir.rglob("*.wav"):
        relative_parts = wav_path.relative_to(cleaned_dir).parts
        # print(relative_parts)     # Structure is ('accents', 'crn_001', 'female', 'lms002', 'shortpassagea_CT.wav')

        if len(relative_parts) < 5:
            print(f">> [WARNING] Unexpected path structure: {relative_parts}")
            continue

        accent = relative_parts[1]
        gender = relative_parts[2]
        speaker_id = relative_parts[3]

        records.append({
            "accent": accent,
            "gender": gender,
            "speaker_id": speaker_id,
            "file_path": str(wav_path.resolve())
        })

    metadata_df = pd.DataFrame(records)

    output_path = cleaned_dir / "metadata.csv"
    metadata_df.to_csv(output_path, index=False)

    print(f">> [OK] Metadata created: {output_path.resolve()}")
    print(f">> [INFO] Number of WAV files: {len(metadata_df)}")
    print(f">> [INFO] Number of speakers: {metadata_df['speaker_id'].nunique()}")

    return metadata_df

In [80]:
metadata_df = create_metadata(OUTPUT_DIR)

>> [OK] Metadata created: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/SpeechProcessing/extracted_wavs/metadata.csv
>> [INFO] Number of WAV files: 855
>> [INFO] Number of speakers: 284


## 4. Load WavLM

In [81]:
processor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)    # Load the processor for the WavLM model.
model = AutoModelForAudioXVector.from_pretrained(MODEL_NAME)    # Load the pre-trained WavLM speaker verification model.

model = model.to(DEVICE)
model.eval()

print(f">> [OK] Model loaded: {MODEL_NAME}")
print(f">> [INFO] Device: {DEVICE}")

Loading weights: 100%|██████████| 266/266 [00:00<00:00, 17399.14it/s]


>> [OK] Model loaded: microsoft/wavlm-base-plus-sv
>> [INFO] Device: cuda


In [82]:
elapsed_time = time.time() - start_time
print(f">> Elapsed Time: {elapsed_time:.3f} seconds | {elapsed_time / 60:.3f} minutes")

>> Elapsed Time: 3.311 seconds | 0.055 minutes
